# OptiCrop: Smart Crop Recommendation Model Development

This notebook contains the complete machine learning development pipeline for **OptiCrop**. 
It performs:
1. Data loading and Exploratory Data Analysis (EDA)
2. Data cleaning & preprocessing (Null checking, duplicates)
3. Feature scaling and label encoding
4. Model training and comparison across 6 classification algorithms
5. Saving the optimal model and artifacts

In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import joblib

## 1. Load and Download Dataset

In [ ]:
dataset_url = "https://raw.githubusercontent.com/Gladiator07/Harvestify/master/Data-processed/crop_recommendation.csv"
dataset_path = "../dataset/Crop_recommendation.csv"

if not os.path.exists(dataset_path):
    os.makedirs(os.path.dirname(dataset_path), exist_ok=True)
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print("Dataset downloaded!")
else:
    print("Dataset exists locally.")

df = pd.read_csv(dataset_path)
print(f"Data Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis & Cleaning

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Missing value check
print("Missing values:")
print(df.isnull().sum())

# Duplicate rows check
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Label distribution
plt.figure(figsize=(12,6))
sns.countplot(y='label', data=df, order=df['label'].value_counts().index, palette='viridis')
plt.title('Distribution of Crop Types')
plt.show()

## 3. Data Preprocessing & Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

# Encode Target Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split Train & Test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed successfully!")

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')
    
    results[name] = {
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
    print(f"{name:20} Accuracy: {acc:.4f}")

In [ ]:
# Convert results to dataframe
results_df = pd.DataFrame(results).T
results_df

## 5. Select & Save the Best Model

In [ ]:
best_model_name = results_df['Accuracy'].idxmax()
best_model = models[best_model_name]
print(f"Best model found: {best_model_name} ({results_df.loc[best_model_name, 'Accuracy']:.4f})")

# Save artifacts
os.makedirs('../model', exist_ok=True)
joblib.dump(best_model, '../model/model.pkl')
joblib.dump(scaler, '../model/scaler.pkl')
joblib.dump(le, '../model/label_encoder.pkl')
print("Best model and encoders serialized and saved!")